# CWSB microscopic camera — FIRST training (one-run Colab)

Use this notebook **once**, to build the first microscopic model (`cwsb_micro_v1`) from the first
batch of microscopic images. For every later batch use `cwsb_retrain_microscopic.ipynb`.

This model is separate from the normal-camera model (`cwsb_v1` / `cwsb_v2`). Do not mix their
images or their model files.

## The one upload

Put one folder (or one zip of it) in Google Drive and set `UPLOAD_PATH` in Cell 1 to where it is,
relative to MyDrive:

```
MICRO_UPLOAD_FOLDER/                 (or MICRO_UPLOAD_FOLDER.zip)
└── new_data/
    ├── healthy/                     microscopic images of NON-infested plants
    │   ├── P001/                    one folder per plant (recommended) ...
    │   │   ├── P001_bark_1.jpg
    │   │   └── P001_bark_2.jpg
    │   └── P002_bark_1.jpg          ... or the plant ID at the start of the file name
    └── infested/                    microscopic images of INFESTED plants
        ├── P101/
        │   ├── P101_hole_1.jpg
        │   └── P101_ridge_1.jpg
        └── P102_hole_1.jpg
```

`old_data/` and `old_models/` do **not** exist yet for the first model. If they are in the upload,
this notebook ignores them.

**Rules for the upload**

- **Class folders** are matched loosely: a folder name containing *unhealthy*, *infected* or
  *infested* counts as infested; any other name containing *health* counts as healthy.
- **Every image needs a plant ID**: `P001`, `P002` … as a folder name or at the start of the file
  name. The split is done by plant, so one plant never lands in both training and testing.
  Images without a plant ID are each treated as their own plant, and the notebook warns you.
- **Use different plant numbers for healthy and infested plants** (a plant is one or the other).
- **Original camera files only**: no crops, no screenshots. Images under 224 px are left out.
- **At least 3 plants per class** are needed to run at all. For a result you can trust, aim for
  about 150 plants per class.

## Before running

Runtime → Change runtime type → **T4 GPU** → Save. Do this first: changing it later wipes the
session. Then Runtime → Run all.

## What it does

1. **Cell 1** settings, GPU check, install.
2. **Cell 2** reads the upload, drops exact duplicates, finds the plant IDs.
3. **Cell 3** quality check, then a 70 / 15 / 15 split by plant.
4. **Cell 4** trains ConvNeXt-Tiny (384 px, 2 classes); prints one line per epoch.
5. **Cell 5** sets the two decision thresholds from the healthy validation scores.
6. **Cell 6** tests on held-out plants, per image and per plant.
7. **Cell 7** writes the heatmap PDF for the test images.
8. **Cell 8** exports ONNX and checks it matches the `.pt` to 4 decimals.
9. **Cell 9** downloads `cwsb_micro_v1_release.zip`.

In [ ]:
# Cell 1 - SETTINGS (the only cell you edit), GPU check, install
VERSION_NUMBER = 1          # <<< outputs become cwsb_micro_v1.pt / .onnx

# Where your upload is, RELATIVE TO MyDrive. It may be a FOLDER or a .zip of that folder.
UPLOAD_PATH = 'cwsb_data/MICRO_UPLOAD_FOLDER.zip'   # <<< edit if yours is somewhere else
USE_DRIVE = True                                    # False = browser-upload the zip instead

# Image quality check. The limits below were set for phone photos, NOT for a microscope,
# so by default they only REPORT. Read the quality table printed by Cell 3, set the
# limits from it, and only then switch to 'exclude'.
#   'report'  -> blurry / over-exposed / dark images are listed but still used
#   'exclude' -> they are left out of training and testing
QUALITY_GATE = 'report'
BLUR_MIN = 60.0        # sharpness score below this  -> "blurry"
CLIPPED_MAX = 0.20     # share of near-white pixels above this -> "clipped" (over-exposed)
DARK_MAX = 0.40        # share of near-black pixels above this -> "dark"
MIN_SIDE = 224         # images with a short side below this many pixels are ALWAYS left out

EPOCHS = 40            # upper limit; training stops by itself when it stops improving
PATIENCE = 6           # stop after this many epochs without improvement

MODE = 'train'      # fixed for this notebook - do not edit

!nvidia-smi -L || echo 'NO GPU - Runtime > Change runtime type > T4 GPU (fix NOW, it wipes files)'
%pip -q install timm onnx onnxruntime onnxscript
import torch
TAG = f'cwsb_micro_v{VERSION_NUMBER}'
print('notebook:', MODE, '| version tag:', TAG, '| torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# Cell 2 - fetch your upload (a Drive folder or one zip) and copy the images into the workspace
import csv, hashlib, json, os, re, shutil, sys, zipfile
from collections import defaultdict
from pathlib import Path

BASE = Path('/content/cwsb')
RAW = BASE / 'Data_final'
shutil.rmtree(BASE, ignore_errors=True)
for d in (RAW / 'Healthy', RAW / 'Unhealthy', BASE / 'src', BASE / 'work'):
    d.mkdir(parents=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    source = Path('/content/drive/MyDrive') / UPLOAD_PATH
    assert source.exists(), f'{source} not found - check UPLOAD_PATH (it is relative to MyDrive)'
else:
    from google.colab import files
    print('upload the zip of your upload folder:')
    up = [n for n in files.upload() if n.lower().endswith('.zip')]
    assert up, 'no zip was uploaded'
    source = Path('/content') / up[0]

if source.is_dir():
    # a plain folder in Drive: read it where it is. Nothing in Drive is changed or deleted.
    STAGE = source
    print(f'reading the folder {source} (images are copied out of Drive, this takes a few minutes)')
else:
    STAGE = Path('/content/_upload')
    shutil.rmtree(STAGE, ignore_errors=True)
    STAGE.mkdir()
    print(f'unpacking {source.name} ({source.stat().st_size / 1e6:.0f} MB) ...')
    with zipfile.ZipFile(source) as zf:
        zf.extractall(STAGE)
    for _ in range(3):                               # zips nested inside the zip
        inner = sorted(STAGE.rglob('*.zip'))
        if not inner:
            break
        for p in inner:
            try:
                with zipfile.ZipFile(p) as zf:
                    zf.extractall(p.with_suffix(''))
            except zipfile.BadZipFile:
                pass
            p.unlink()

# --- locate old_data / new_data / old_models (directly in the upload or under one parent folder)
def find_dir(name):
    hits = [p for p in STAGE.rglob('*') if p.is_dir() and p.name.lower() == name]
    return min(hits, key=lambda p: len(p.parts)) if hits else None

OLD_DIR, NEW_DIR, MODEL_DIR = find_dir('old_data'), find_dir('new_data'), find_dir('old_models')
assert NEW_DIR, 'no new_data/ folder found in your upload - it is required'
if MODE == 'retrain':
    assert OLD_DIR, ('no old_data/ folder found in your upload. Retraining needs old_data/ + new_data/. '
                     'For the very first model use cwsb_train_microscopic.ipynb')
else:
    if OLD_DIR or MODEL_DIR:
        print('NOTE: old_data/ and old_models/ are ignored by the first-training notebook. '
              'To use them, run cwsb_retrain_microscopic.ipynb')
    OLD_DIR = MODEL_DIR = None

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
PLANT_RE = re.compile(r'(?:^|[^a-z0-9])(?:plant|p)[-_ ]?(\d{1,5})(?![0-9])', re.IGNORECASE)
manifest, skipped, mismatch, cross_dup = [], [], [], []
seen_hash, used_names = {}, set()

def label_of(text):
    """Folder name -> class. 'unhealthy', 'infected', 'infested' -> Unhealthy; 'healthy' -> Healthy."""
    s = text.lower()
    if 'unhealthy' in s or 'infect' in s or 'infest' in s:
        return 'Unhealthy'
    if 'health' in s:
        return 'Healthy'
    return None

def plant_of(folders, stem):
    """Plant ID from the nearest folder name that has one (P001, plant_12 ...), else from the file name."""
    for i, text in enumerate(list(reversed(folders)) + [stem]):
        m = PLANT_RE.search(text)
        if m:
            return f'P{int(m.group(1)):04d}', i < len(folders)
    return '', False

def ingest(root, tag):
    stats = {'files': 0, 'kept': 0, 'duplicates': 0}
    for p in sorted(Path(root).rglob('*')):
        if p.suffix.lower() not in IMG_EXTS or p.name.startswith('._') or '__MACOSX' in p.parts:
            continue
        stats['files'] += 1
        rel = p.relative_to(root)
        folders = list(rel.parts[:-1])
        label = label_of('/'.join(folders))
        if label is None:
            skipped.append(f'{tag}_data/{rel.as_posix()}')
            continue
        said = label_of(p.stem)
        if said and said != label:                   # folder says one class, file name says the other
            mismatch.append(f'{tag}_data/{rel.as_posix()}')
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
        if digest in seen_hash:                      # byte-identical copy of an image already taken
            stats['duplicates'] += 1
            if seen_hash[digest][0] != label:
                cross_dup.append((digest, f'{tag}_data/{rel.as_posix()}  ==  {seen_hash[digest][1]}'))
            continue
        seen_hash[digest] = (label, f'{tag}_data/{rel.as_posix()}')
        plant, from_folder = plant_of(folders, p.stem)
        stem = re.sub(r'[^\w.-]+', '_', p.stem)
        if from_folder and plant_of([], p.stem)[0] != plant:
            stem = f'{plant}_{stem}'               # two plants may both have an 'img1.jpg'
        if (label, stem.lower()) in used_names:
            stem = f'{stem}_{digest[:8]}'
        used_names.add((label, stem.lower()))
        name = f'{label}/{stem}{p.suffix.lower()}'
        shutil.copy(p, RAW / name)
        manifest.append({'file': name, 'label': label, 'plant': plant, 'source': tag, 'sha256': digest})
        stats['kept'] += 1
    print(f"{tag}_data: {stats['files']} image files -> {stats['kept']} kept, "
          f"{stats['duplicates']} exact duplicates dropped")

if OLD_DIR:
    ingest(OLD_DIR, 'old')
ingest(NEW_DIR, 'new')

# the same file in a healthy AND an infested folder: its label is unknown, so it is left out
unsure = {d for d, _ in cross_dup}
for m in [m for m in manifest if m['sha256'] in unsure]:
    (RAW / m['file']).unlink()
    manifest.remove(m)

with open(BASE / 'work' / 'manifest.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['file', 'label', 'plant', 'source', 'sha256'])
    w.writeheader()
    w.writerows(manifest)
(BASE / 'work' / 'settings.json').write_text(json.dumps({
    'quality_gate': QUALITY_GATE, 'blur_min': BLUR_MIN, 'clipped_max': CLIPPED_MAX,
    'dark_max': DARK_MAX, 'min_side': MIN_SIDE}, indent=2))

# --- the previous model and its split file (retrain only)
OLD_PT = PREV_SPLIT = None
if MODEL_DIR:
    pts = sorted(MODEL_DIR.rglob('*.pt'))
    OLD_PT = str(pts[0]) if pts else None
    splits = sorted(p for p in MODEL_DIR.rglob('*.csv') if 'split' in p.name.lower())
    if splits:
        PREV_SPLIT = splits[0]
        shutil.copy(PREV_SPLIT, BASE / 'work' / 'prev_split.csv')
if MODE == 'retrain':
    print('old model for comparison      :', OLD_PT or 'NONE - the old-vs-new comparison will be skipped')
    print('previous split file (split csv):', PREV_SPLIT.name if PREV_SPLIT else
          'NONE - every image is split afresh, so the old-vs-new comparison will NOT be fair')

print(f"\n  {'':10s} {'healthy':>20s} {'infested':>20s}")
for tag in ('old', 'new'):
    cell = []
    for label in ('Healthy', 'Unhealthy'):
        rs = [m for m in manifest if m['source'] == tag and m['label'] == label]
        cell.append(f"{len(rs):5d} ({len({m['plant'] for m in rs if m['plant']}):3d} plants)")
    if tag == 'new' or OLD_DIR:
        print(f'  {tag + "_data":10s} {cell[0]:>20s} {cell[1]:>20s}')
no_id = sum(not m['plant'] for m in manifest)
if no_id:
    print(f'WARNING: {no_id} of {len(manifest)} images have no plant ID (P001 ... in the folder or file name)')
if skipped:
    print(f'WARNING: {len(skipped)} images skipped - they are not inside a healthy / infested folder:')
    for s in skipped[:10]:
        print('  ', s)
if mismatch:
    print(f'WARNING: {len(mismatch)} images sit in one class folder but their file name says the other class.'
          ' The FOLDER was used. Please check:')
    for s in mismatch[:10]:
        print('  ', s)
if cross_dup:
    print(f'WARNING: {len(unsure)} images are in BOTH a healthy and an infested folder (same file). '
          'They are LEFT OUT until the label is fixed:')
    for _, s in cross_dup[:10]:
        print('  ', s)
n_h = sum(m['label'] == 'Healthy' for m in manifest)
n_u = len(manifest) - n_h
assert n_h > 0 and n_u > 0, 'one class has no images - check the folder layout in the first cell'
if MODE == 'retrain' and not any(m['source'] == 'new' for m in manifest):
    print('WARNING: new_data/ added no new images (empty, or all copies of old_data). '
          'The new model will be trained on the old images only.')

## The pipeline scripts

The next three cells only **write** the pipeline scripts into the session; they run in Cells 3
and 4. Nothing needs editing here.

In [ ]:
%%writefile /content/cwsb/src/clean_micro.py
"""Microscopic pipeline, step 1 - clean the uploaded images into work/clean.

Per image: apply EXIF rotation, strip ALL metadata (GPS!), save a plain JPEG q95
at full resolution, and record quality scores (blur / clipped / dark / size) and
a perceptual dhash. Then report exact-looking and near duplicates.

Differences from the normal-camera clean.py:
  * reads work/manifest.csv (written by the notebook), so every image carries
    its plant ID and whether it came from old_data or new_data;
  * the image is only ever SHRUNK to 1024 px wide before the blur score, never
    enlarged (enlarging a small image makes it look blurry);
  * the quality limits come from the notebook's Cell 1 (work/settings.json).
"""
import csv
import json
import sys
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageOps

ROOT = Path(__file__).resolve().parents[1]
RAW = ROOT / "Data_final"
WORK = ROOT / "work"
DST = WORK / "clean"
NEAR_DUP_HAMMING = 6   # dhash distance <= this => near duplicate

FIELDS = ["file", "label", "plant", "source", "sha256", "width", "height",
          "blur", "clipped", "dark", "dhash", "flags"]


def dhash(gray, size=8):
    small = cv2.resize(gray, (size + 1, size), interpolation=cv2.INTER_AREA)
    bits = small[:, 1:] > small[:, :-1]
    return int("".join("1" if b else "0" for b in bits.flatten()), 2)


def pct(values, q):
    return float(np.percentile(values, q)) if len(values) else float("nan")


def main():
    cfg = json.loads((WORK / "settings.json").read_text())
    with open(WORK / "manifest.csv", newline="") as f:
        manifest = list(csv.DictReader(f))
    print(f"{len(manifest)} images to clean")

    rows = []
    for i, m in enumerate(manifest):
        src = RAW / m["file"]
        out_rel = f"{m['label']}/{Path(m['file']).stem}.jpg"
        row = {"file": out_rel, "label": m["label"], "plant": m["plant"],
               "source": m["source"], "sha256": m["sha256"], "width": 0, "height": 0,
               "blur": 0.0, "clipped": 0.0, "dark": 0.0, "dhash": "", "flags": ""}
        try:
            img = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
        except Exception as e:  # a broken file must not stop the run
            row["flags"] = "unreadable"
            rows.append(row)
            print(f"  unreadable: {m['file']} ({type(e).__name__})")
            continue
        out = DST / out_rel
        out.parent.mkdir(parents=True, exist_ok=True)
        img.save(out, "JPEG", quality=95, exif=b"")   # strips GPS/EXIF

        g = cv2.cvtColor(np.asarray(img), cv2.COLOR_RGB2GRAY)
        if g.shape[1] > 1024:                           # shrink only, never enlarge
            g = cv2.resize(g, (1024, max(int(g.shape[0] * 1024 / g.shape[1]), 1)),
                           interpolation=cv2.INTER_AREA)
        blur = float(cv2.Laplacian(g, cv2.CV_64F).var())
        clipped = float((g > 250).mean())
        dark = float((g < 15).mean())

        flags = []
        if min(img.size) < cfg["min_side"]:
            flags.append("tiny")
        if blur < cfg["blur_min"]:
            flags.append("blurry")
        if clipped > cfg["clipped_max"]:
            flags.append("clipped")
        if dark > cfg["dark_max"]:
            flags.append("dark")
        row.update(width=img.width, height=img.height, blur=round(blur, 2),
                   clipped=round(clipped, 4), dark=round(dark, 4),
                   dhash=f"{dhash(g):016x}", flags="|".join(flags))
        rows.append(row)
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(manifest)}")

    DST.mkdir(parents=True, exist_ok=True)
    with open(DST / "images.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows(rows)

    # --- near-duplicate report (same picture saved twice, or two labels on one picture)
    ok = [r for r in rows if r["dhash"]]
    hashes = [int(r["dhash"], 16) for r in ok]
    dupes = []
    for i in range(len(ok)):
        for j in range(i + 1, len(ok)):
            d = bin(hashes[i] ^ hashes[j]).count("1")
            if d <= NEAR_DUP_HAMMING:
                dupes.append((ok[i]["file"], ok[j]["file"], d))
    with open(DST / "duplicates.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["file_a", "file_b", "hamming"])
        w.writerows(dupes)
    cross = [d for d in dupes if d[0].split("/")[0] != d[1].split("/")[0]]

    # --- what the quality numbers look like, so the limits in Cell 1 can be set from real data
    print(f"\nDone: {len(rows)} images -> {DST}")
    print("\nQuality scores per class (use these to set the limits in Cell 1):")
    print(f"  {'class':10s} {'n':>5s}  {'size (short side) min/median':>29s}   blur p5 / p25 / median")
    for label in ("Healthy", "Unhealthy"):
        rs = [r for r in rows if r["label"] == label and r["dhash"]]
        if not rs:
            continue
        sides = [min(r["width"], r["height"]) for r in rs]
        blurs = [r["blur"] for r in rs]
        print(f"  {label:10s} {len(rs):5d}  {min(sides):14d} / {int(np.median(sides)):<12d}   "
              f"{pct(blurs, 5):.0f} / {pct(blurs, 25):.0f} / {pct(blurs, 50):.0f}")

    print(f"\nFlags with the current limits (blur < {cfg['blur_min']}, clipped > {cfg['clipped_max']}, "
          f"dark > {cfg['dark_max']}, short side < {cfg['min_side']} px):")
    for flag in ("unreadable", "tiny", "blurry", "clipped", "dark"):
        hit = [r for r in rows if flag in r["flags"].split("|")]
        if hit:
            per = {lab: sum(r["label"] == lab for r in hit) for lab in ("Healthy", "Unhealthy")}
            print(f"  {flag:10s} {len(hit):4d}  (healthy {per['Healthy']}, infested {per['Unhealthy']})")
    if not any(r["flags"] for r in rows):
        print("  none")
    if cfg["quality_gate"] == "exclude":
        print("QUALITY_GATE = 'exclude': every flagged image is left out of training and testing.")
    else:
        print("QUALITY_GATE = 'report': only 'tiny' and 'unreadable' images are left out; "
              "blurry / clipped / dark are reported but still used.")

    same = len(dupes) - len(cross)
    print(f"\nNear-duplicate pairs: {len(dupes)} ({same} within one class, {len(cross)} ACROSS classes)")
    for d in cross[:20]:
        print(f"  CHECK THE LABEL: {d[0]} <-> {d[1]} (distance {d[2]})")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /content/cwsb/src/split_micro.py
"""Microscopic pipeline, step 2 - leak-proof train / val / test split BY PLANT.

Every image of one plant goes to the same set, so the test set only ever holds
plants the model has never seen. 70 / 15 / 15 per class, seed 42.

Differences from the normal-camera make_split.py:
  * the plant comes from the plant ID in the folder or file name (P001 ...),
    not from guessing by frame number and time gap;
  * RETRAIN: if work/prev_split.csv exists (the previous version's split file),
    every image that was in it keeps the set it had. Only the new images are
    split. So the old test images stay test images, and the old model can be
    compared with the new one on images that neither of them trained on.
"""
import csv
import json
import random
import sys
from collections import defaultdict
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
WORK = ROOT / "work"
CLEAN = WORK / "clean"
OUT = WORK / "splits" / "split.csv"
PREV = WORK / "prev_split.csv"

FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED = 42
ALWAYS_OUT = {"tiny", "unreadable"}     # left out whatever the quality gate says
FIELDS = ["file", "label", "group", "split", "source", "sha256"]


def main():
    cfg = json.loads((WORK / "settings.json").read_text())
    with open(CLEAN / "images.csv", newline="") as f:
        rows = list(csv.DictReader(f))

    for r in rows:
        flags = set(filter(None, r["flags"].split("|")))
        r["out"] = bool(flags) if cfg["quality_gate"] == "exclude" else bool(flags & ALWAYS_OUT)

    # --- the previous version's split, matched by image content (so renaming a file is harmless)
    prev = {}
    if PREV.exists():
        with open(PREV, newline="") as f:
            for p in csv.DictReader(f):
                if p.get("sha256") and p.get("split") in FRACTIONS:
                    prev[p["sha256"]] = (p["split"], p["group"])
        print(f"previous split: {len(prev)} images with a fixed set")

    # --- one group per plant
    no_id = 0
    for r in rows:
        if r["sha256"] in prev:
            r["group"] = prev[r["sha256"]][1]
        elif r["plant"]:
            r["group"] = f"{r['label']}/{r['plant']}"
        else:
            r["group"] = f"{r['label']}/~{Path(r['file']).stem}"   # no plant ID: alone in its group
            no_id += 1

    frozen, clash = {}, set()
    for r in rows:
        if r["sha256"] in prev and not r["out"]:
            s = prev[r["sha256"]][0]
            if frozen.setdefault(r["group"], s) != s:
                clash.add(r["group"])
    if clash:
        print(f"WARNING: {len(clash)} plants were in two sets in the previous split; kept the first")

    groups = defaultdict(list)
    for r in rows:
        if not r["out"]:
            groups[r["group"]].append(r)

    # --- split the plants that have no fixed set yet, per class
    rng = random.Random(SEED)
    assign = dict(frozen)
    for label in ("Healthy", "Unhealthy"):
        mine = sorted(g for g in groups if groups[g][0]["label"] == label)
        total = sum(len(groups[g]) for g in mine)
        want = {s: total * frac for s, frac in FRACTIONS.items()}
        got = {s: 0 for s in FRACTIONS}
        todo = []
        for g in mine:
            if g in assign:
                got[assign[g]] += len(groups[g])
            else:
                todo.append(g)
        rng.shuffle(todo)
        for s in ("test", "val", "train"):          # every set gets at least one plant of the class
            if got[s] == 0 and todo:
                g = todo.pop(0)
                assign[g] = s
                got[s] += len(groups[g])
        for g in todo:
            s = max(FRACTIONS, key=lambda k: want[k] - got[k])
            assign[g] = s
            got[s] += len(groups[g])

    for r in rows:
        r["split"] = "excluded" if r["out"] else assign[r["group"]]

    # --- leakage checks
    plant_sets = defaultdict(set)
    for r in rows:
        if r["split"] != "excluded":
            plant_sets[r["group"]].add(r["split"])
    leaks = {g: s for g, s in plant_sets.items() if len(s) > 1}
    if leaks:
        print(f"LEAKAGE: {leaks}")
        return 1

    dup_leaks = []
    dup_file = CLEAN / "duplicates.csv"
    if dup_file.exists():
        split_of = {r["file"]: r["split"] for r in rows}
        with open(dup_file, newline="") as f:
            for d in csv.DictReader(f):
                sa, sb = split_of.get(d["file_a"]), split_of.get(d["file_b"])
                if sa and sb and sa != sb and "excluded" not in (sa, sb):
                    dup_leaks.append((d["file_a"], sa, d["file_b"], sb))
    for d in dup_leaks[:10]:
        print(f"WARNING near-duplicate images in two sets: {d}")

    OUT.parent.mkdir(parents=True, exist_ok=True)
    with open(OUT, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows([{k: r[k] for k in FIELDS} for r in rows])

    # --- report
    with_id = len(rows) - no_id
    print(f"images with a plant ID: {with_id} of {len(rows)}")
    if no_id:
        print(f"WARNING: {no_id} images have no plant ID (P001 ... in the folder or file name).\n"
              "         Each one is treated as its own plant. If several of them show the same\n"
              "         plant, that plant can be in training AND testing and the test result\n"
              "         will look better than it is.")
    print(f"\n  {'set':9s} {'healthy':>16s} {'infested':>16s} {'total':>7s}")
    counts = {}
    for s in ("train", "val", "test", "excluded"):
        cell = {}
        for label in ("Healthy", "Unhealthy"):
            rs = [r for r in rows if r["split"] == s and r["label"] == label]
            cell[label] = (len(rs), len({r["group"] for r in rs}))
        counts[s] = cell
        h, u = cell["Healthy"], cell["Unhealthy"]
        print(f"  {s:9s} {h[0]:5d} ({h[1]:3d} plants) {u[0]:5d} ({u[1]:3d} plants) {h[0] + u[0]:7d}")
    if prev:
        print("\n  of which, by where the image came from:")
        for s in ("train", "val", "test"):
            old = sum(r["split"] == s and r["source"] == "old" for r in rows)
            new = sum(r["split"] == s and r["source"] == "new" for r in rows)
            kept = sum(r["split"] == s and r["sha256"] in prev for r in rows)
            print(f"  {s:9s} old_data {old:5d}   new_data {new:5d}   (kept from the previous split: {kept})")

    for s in ("train", "val"):
        for label in ("Healthy", "Unhealthy"):
            if counts[s][label][0] == 0:
                print(f"ERROR: the {s} set has no {label} images - at least 3 plants per class are needed")
                return 1
    for label in ("Healthy", "Unhealthy"):
        if counts["test"][label][0] == 0:
            print(f"WARNING: the test set has no {label} images")
    print("LEAKAGE CHECK OK" + (f" ({len(dup_leaks)} near-duplicate warnings)" if dup_leaks else ""))
    print(f"-> {OUT}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /content/cwsb/src/train_micro.py
"""Microscopic pipeline, step 3 - train the CWSB microscopic classifier.

Same model and recipe as the normal-camera model (contender B):
ConvNeXt-Tiny at 384 px, ImageNet weights, 2 warm-up epochs on the head and then
all layers, weighted sampling + class-weighted loss, early stopping.
Positive class = Unhealthy (infested). Model selection = validation PR-AUC. Seed 42.

Differences from the normal-camera train.py:
  * no healthy tiles (they existed only to patch the 28-vs-470 imbalance);
  * vertical flips are added, because a microscope view has no "up";
  * the per-epoch history is saved in training_result.json.

Usage (from the project root, after clean + split):
  python src/train_micro.py --epochs 40 --patience 6
"""
import argparse
import csv
import json
import os
import random
from pathlib import Path

import numpy as np
import timm
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import average_precision_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms

ROOT = Path(__file__).resolve().parents[1]
CLEAN = ROOT / "work" / "clean"
SPLIT = ROOT / "work" / "splits" / "split.csv"
OUT = ROOT / "models" / "micro"

IMG = 384
BACKBONE = "convnext_tiny.fb_in22k_ft_in1k_384"
SEED = 42
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
WORKERS = 0 if os.name == "nt" else 2

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.10),
    transforms.RandomApply([transforms.GaussianBlur(5)], p=0.15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([  # MUST stay identical wherever the model is served
    transforms.Resize(IMG + 32),
    transforms.CenterCrop(IMG),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class CwsbDataset(Dataset):
    def __init__(self, rows, tf):
        self.rows, self.tf = rows, tf

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        img = Image.open(CLEAN / r["file"]).convert("RGB")
        return self.tf(img), 1 if r["label"] == "Unhealthy" else 0


def load_rows(split):
    with open(SPLIT, newline="") as f:
        return [r for r in csv.DictReader(f) if r["split"] == split]


def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


@torch.no_grad()
def scores(model, rows, device, bs=16):
    model.eval()
    dl = DataLoader(CwsbDataset(rows, eval_tf), batch_size=bs, num_workers=WORKERS)
    ys, ps = [], []
    for x, y in dl:
        p = torch.softmax(model(x.to(device)), 1)[:, 1]
        ps += p.cpu().tolist()
        ys += y.tolist()
    return np.array(ys), np.array(ps)


def main():
    ap_ = argparse.ArgumentParser()
    ap_.add_argument("--epochs", type=int, default=40)
    ap_.add_argument("--patience", type=int, default=6)
    args = ap_.parse_args()
    set_seed()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device: {device}")

    model = timm.create_model(BACKBONE, pretrained=True, num_classes=2).to(device)
    tr, va = load_rows("train"), load_rows("val")
    n_pos = sum(r["label"] == "Unhealthy" for r in tr)
    n_neg = len(tr) - n_pos
    print(f"train {len(tr)} ({n_neg} healthy, {n_pos} infested), val {len(va)}")
    assert n_pos and n_neg, "the training set needs both classes"

    weights = [1.0 / (n_pos if r["label"] == "Unhealthy" else n_neg) for r in tr]
    sampler = WeightedRandomSampler(weights, num_samples=len(tr), replacement=True)
    dl = DataLoader(CwsbDataset(tr, train_tf), batch_size=16, sampler=sampler, num_workers=WORKERS)
    cw = torch.tensor([len(tr) / (2 * n_neg), len(tr) / (2 * n_pos)], dtype=torch.float32)
    crit = nn.CrossEntropyLoss(weight=cw.to(device))

    head = [p for n, p in model.named_parameters() if "head" in n]
    body = [p for n, p in model.named_parameters() if "head" not in n]

    best_ap, best_ep, since, history = -1.0, -1, 0, []
    OUT.mkdir(parents=True, exist_ok=True)
    for ep in range(args.epochs):
        warm = ep < 2
        for p in body:
            p.requires_grad = not warm
        opt = torch.optim.AdamW(
            [{"params": head, "lr": 1e-3 if warm else 1e-4},
             {"params": body, "lr": 0 if warm else 1e-4}], weight_decay=0.05)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=len(dl))
        model.train()
        tot = 0.0
        for x, y in dl:
            opt.zero_grad()
            loss = crit(model(x.to(device)), y.to(device))
            loss.backward()
            opt.step()
            sched.step()
            tot += loss.item()
        yva, pva = scores(model, va, device)
        ap = float(average_precision_score(yva, pva))
        history.append({"epoch": ep, "phase": "warm" if warm else "full",
                        "loss": round(tot / len(dl), 4), "val_pr_auc": round(ap, 5)})
        print(f"ep{ep:02d} {'warm' if warm else 'full'}: loss {tot / len(dl):.4f}  val PR-AUC {ap:.4f}"
              + ("  <- best so far, saved" if ap > best_ap else ""))
        if ap > best_ap:
            best_ap, best_ep, since = ap, ep, 0
            torch.save({"mode": "B", "backbone": BACKBONE, "img": IMG,
                        "state_dict": model.state_dict(),
                        "val_pr_auc": ap, "epoch": ep}, OUT / "model.pt")
        else:
            since += 1
            if since >= args.patience:
                print(f"early stop: no improvement for {args.patience} epochs")
                break

    (OUT / "training_result.json").write_text(json.dumps(
        {"val_pr_auc": best_ap, "best_epoch": best_ep, "epochs_run": len(history),
         "train_images": len(tr), "train_healthy": n_neg, "train_infested": n_pos,
         "val_images": len(va), "backbone": BACKBONE, "img": IMG, "seed": SEED,
         "history": history}, indent=2))
    print(f"\nran {len(history)} epochs; best = epoch {best_ep} with val PR-AUC {best_ap:.4f}")


if __name__ == "__main__":
    main()

In [ ]:
# Cell 3 - quality check -> split by plant (the two scripts written by the cells above)
import subprocess
os.chdir(BASE)

def run(script, *args):
    """Run a pipeline script and show its output line by line as it happens."""
    print('\n' + '=' * 25, script, '=' * 25)
    proc = subprocess.Popen([sys.executable, '-u', script, *map(str, args)], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
    for line in proc.stdout:
        print(line, end='')
    assert proc.wait() == 0, script + ' FAILED - read its output above before rerunning'

run('src/clean_micro.py')
run('src/split_micro.py')
print('\ndata stages done')

In [ ]:
# Cell 4 - train the ConvNeXt-Tiny classifier. One line per epoch is printed below.
run('src/train_micro.py', '--epochs', EPOCHS, '--patience', PATIENCE)
assert Path('models/micro/model.pt').exists(), 'training failed'
print('training done -> models/micro/model.pt (renamed at packaging time)')

In [ ]:
# Cell 5 - decision thresholds, set from the HEALTHY validation scores
import numpy as np, timm
sys.path.insert(0, str(BASE / 'src'))
import train_micro as Tr

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ck = torch.load('models/micro/model.pt', map_location=device, weights_only=False)
new_model = timm.create_model(ck['backbone'], pretrained=False, num_classes=2).to(device)
new_model.load_state_dict(ck['state_dict'])
new_model.eval()
print(f"new model: epoch {ck['epoch']}  val PR-AUC {float(ck['val_pr_auc']):.4f}")

va = Tr.load_rows('val')
yv, pv = Tr.scores(new_model, va, device)
healthy_val, infected_val = pv[yv == 0], pv[yv == 1]
print(f'val healthy  n={len(healthy_val)}  scores {healthy_val.min():.3f} - {healthy_val.max():.3f}')
print(f'val infested n={len(infected_val)}  scores {infected_val.min():.3f} - {infected_val.max():.3f}')

T_HIGH = 0.90
if len(healthy_val) >= 5:
    T_LOW = float(np.clip(round(float(healthy_val.max()) + 0.05, 2), 0.35, 0.60))
    basis = f'max healthy val score {healthy_val.max():.3f} + 0.05 margin, clamped to [0.35, 0.60]'
else:
    T_LOW = 0.50
    basis = f'DEFAULT - only {len(healthy_val)} healthy val images, too few to calibrate on'
    print('WARNING:', basis)
print(f'T_low = {T_LOW}   T_high = {T_HIGH}   ({basis})')

config = {
    'model': f'{TAG}.pt', 'onnx': f'{TAG}.onnx', 'camera': 'microscopic',
    'backbone': ck['backbone'], 'img': ck['img'], 'version': VERSION_NUMBER,
    'T_low': T_LOW, 'T_high': T_HIGH,
    'plant_score': "max over the plant's images",
    'answers': {'score >= T_high': 'wsb_likely',
                'T_low <= score < T_high': 'inconclusive',
                'score < T_low': 'no_visible_signs'},
    'threshold_basis': basis,
    'calibrated_on': 'validation split only; test split used as a check',
    'quality_gate': {'mode': QUALITY_GATE, 'blur_min': BLUR_MIN, 'clipped_max': CLIPPED_MAX,
                     'dark_max': DARK_MAX, 'min_side': MIN_SIDE},
}
Path('models/micro/config.json').write_text(json.dumps(config, indent=2))
print('-> models/micro/config.json')

In [ ]:
# Cell 6 - held-out TEST evaluation (plants the model never saw), per image and per plant
te = Tr.load_rows('test')
yt, pt_new = Tr.scores(new_model, te, device)
plants = sorted({r['group'] for r in te})
row_plant = np.array([plants.index(r['group']) for r in te])
row_new = np.array([r['source'] == 'new' for r in te])

def tally(name, y, p):
    v = np.where(p >= T_HIGH, 'wsb', np.where(p < T_LOW, 'clear', 'inc'))
    caught = int(((y == 1) & (v == 'wsb')).sum());   missed = int(((y == 1) & (v == 'clear')).sum())
    fa = int(((y == 0) & (v == 'wsb')).sum());       cleared = int(((y == 0) & (v == 'clear')).sum())
    inc = int((v == 'inc').sum())
    rec = f'{caught / (caught + missed):.1%}' if caught + missed else '  n/a'
    spec = f'{cleared / (cleared + fa):.1%}' if cleared + fa else '  n/a'
    print(f'  {name:26s} n={len(y):4d}  caught {caught:4d}  missed {missed:3d}  false-alarms {fa:3d}  '
          f'cleared {cleared:4d}  inconclusive {inc:3d}  recall {rec}  correct-clears {spec}')

def per_plant(y, p, keep):
    """One score per plant = its highest image score (how the device decides)."""
    ids = sorted(set(row_plant[keep]))
    return (np.array([y[keep & (row_plant == i)][0] for i in ids]),
            np.array([p[keep & (row_plant == i)].max() for i in ids]))

def report(name, p):
    everything = np.ones(len(te), dtype=bool)
    tally(f'{name} - images', yt, p)
    tally(f'{name} - plants', *per_plant(yt, p, everything))
    if MODE == 'retrain' and row_new.any() and (~row_new).any():
        tally(f'{name} - old_data images', yt[~row_new], p[~row_new])
        tally(f'{name} - new_data images', yt[row_new], p[row_new])

print(f'TEST set: {len(te)} images from {len(plants)} plants, thresholds {T_LOW} / {T_HIGH}')
report(f'NEW {TAG}', pt_new)

if MODE == 'retrain':
    if OLD_PT:
        cko = torch.load(OLD_PT, map_location=device, weights_only=False)
        sd = cko.get('state_dict', cko) if isinstance(cko, dict) else cko
        old_model = timm.create_model(cko.get('backbone', ck['backbone']) if isinstance(cko, dict) else ck['backbone'],
                                      pretrained=False, num_classes=2).to(device)
        old_model.load_state_dict({k.replace('module.', ''): v for k, v in sd.items()})
        old_model.eval()
        _, pt_old = Tr.scores(old_model, te, device)
        print()
        report('OLD ' + Path(OLD_PT).stem, pt_old)
        del old_model
        if PREV_SPLIT:
            print('\nThe old images kept the set they had in the previous split, so NEITHER model trained on\n'
                  'any test image. This is a fair comparison. Keep the old model unless the new one wins.')
        else:
            print('\nWARNING: no previous split file was in old_models/, so some of these test images were\n'
                  '         TRAINING images of the old model. Its numbers here are too good. Do not use this\n'
                  '         table to say the new model is worse.')
    else:
        print('\n(no .pt in old_models/ - old-vs-new comparison skipped)')

In [ ]:
# Cell 7 - Grad-CAM heatmap PDF for the TEST images: where the new model looked, failures first
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch.nn.functional as F
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image
from torchvision import transforms

MAX_PDF_PAGES = 200         # wrong and inconclusive images always come first
IMG = ck['img']
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
eval_tf = transforms.Compose([transforms.Resize(IMG + 32), transforms.CenterCrop(IMG),
                              transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
disp_tf = transforms.Compose([transforms.Resize(IMG + 32), transforms.CenterCrop(IMG)])


def score_and_cam(model, pil):
    """One forward + backward: (prob_infested, IMGxIMG heatmap 0-1)."""
    store = {}

    def grab(module, inp, out):
        out.retain_grad()
        store['a'] = out

    h = model.stages[-1].register_forward_hook(grab)
    x = eval_tf(pil).unsqueeze(0).to(device)
    logits = model(x)
    prob = torch.softmax(logits, 1)[0, 1].item()
    model.zero_grad()
    logits[0, 1].backward()
    h.remove()

    a = store['a']
    w = a.grad.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(IMG, IMG), mode='bilinear', align_corners=False)
    cam = cam[0, 0].detach().cpu().numpy()
    rng = cam.max() - cam.min()
    return prob, (cam - cam.min()) / rng if rng > 1e-9 else np.zeros_like(cam)


def verdict_of(s):
    return ('WSB SIGNS LIKELY' if s >= T_HIGH else
            'No visible WSB signs' if s < T_LOW else 'INCONCLUSIVE - retake')


recs = []
for r in te:
    pil = Image.open(Path('work/clean') / r['file']).convert('RGB')
    s, cam = score_and_cam(new_model, pil)
    v = verdict_of(s)
    mark = ('inconclusive' if v.startswith('INCONCL')
            else 'CORRECT' if (v.startswith('WSB') == (r['label'] == 'Unhealthy')) else 'WRONG')
    recs.append({'file': r['file'], 'truth': r['label'], 'plant': r['group'], 'score': s,
                 'verdict': v, 'mark': mark, 'disp': np.array(disp_tf(pil)), 'cam': cam})

dec = [r for r in recs if r['mark'] != 'inconclusive']
acc = sum(r['mark'] == 'CORRECT' for r in dec) / max(len(dec), 1)
inf = [r for r in dec if r['truth'] == 'Unhealthy']
hea = [r for r in dec if r['truth'] == 'Healthy']
rec_ = sum(r['verdict'].startswith('WSB') for r in inf) / max(len(inf), 1)
spec = sum(not r['verdict'].startswith('WSB') for r in hea) / max(len(hea), 1)

Path('reports').mkdir(exist_ok=True)
pdf_path = Path('reports') / f'{TAG}_test_report.pdf'
order = ([r for r in recs if r['mark'] == 'WRONG'] +
         [r for r in recs if r['mark'] == 'inconclusive'] +
         [r for r in recs if r['mark'] == 'CORRECT'])[:MAX_PDF_PAGES]

with PdfPages(pdf_path) as pdf:
    L = [f'CWSB MICROSCOPIC {TAG} - HELD-OUT TEST REPORT', '',
         f'Backbone        : {ck["backbone"]}',
         f'Thresholds      : no signs < {T_LOW}  |  inconclusive  |  >= {T_HIGH} WSB likely',
         f'Test images     : {len(recs)} from {len(plants)} plants  '
         f'({len(hea)} healthy, {len(inf)} infested, decided)',
         '', 'These plants were never used for training or threshold tuning.', '',
         '--- RESULTS (per image) ---', '',
         f'  Recall on infested     : {rec_:6.1%}',
         f'  Correct clears         : {spec:6.1%}',
         f'  Accuracy where decided : {acc:6.1%}',
         f'  Sent to retake         : {sum(r["mark"] == "inconclusive" for r in recs)} of {len(recs)}',
         f'  Wrong                  : {sum(r["mark"] == "WRONG" for r in recs)} of {len(recs)}',
         f'  Pages that follow      : {len(order)} of {len(recs)} images',
         '', '--- FAILURES (pages follow first) ---', '']
    wrong = sorted([x for x in recs if x['mark'] == 'WRONG'], key=lambda x: x['score'])
    for r in wrong[:25]:
        kind = 'false alarm' if r['truth'] == 'Healthy' else 'missed infestation'
        L.append(f'  {r["file"][:46]:46s} {r["score"]:.3f}  {kind}')
    if len(wrong) > 25:
        L.append(f'  ... and {len(wrong) - 25} more')
    if not wrong:
        L.append('  none')
    fig = plt.figure(figsize=(10, 7.5))
    fig.text(0.05, 0.96, chr(10).join(L), va='top', family='monospace', fontsize=8.5)
    plt.axis('off'); pdf.savefig(fig, dpi=90); plt.close(fig)

    for r in order:
        col = {'CORRECT': '#2e7d32', 'WRONG': '#b3261e', 'inconclusive': '#a05a00'}[r['mark']]
        fig, ax = plt.subplots(1, 2, figsize=(10, 5.4))
        ax[0].imshow(r['disp']); ax[0].set_title('image as the model saw it', fontsize=10)
        ax[1].imshow(r['disp']); ax[1].imshow(r['cam'], cmap='jet', alpha=0.45)
        ax[1].set_title('what drove the decision (red = most)', fontsize=10)
        for a in ax:
            a.axis('off')
        fig.suptitle(f'{r["file"]}   true: {r["truth"]}   score {r["score"]:.3f}  ->  '
                     f'{r["verdict"]}   [{r["mark"]}]   plant {r["plant"]}', color=col, fontsize=10)
        pdf.savefig(fig, dpi=80); plt.close(fig)

print(f'recall {rec_:.1%} | correct clears {spec:.1%} | accuracy where decided {acc:.1%}')
print(f'-> {pdf_path}  ({pdf_path.stat().st_size / 1e6:.1f} MB, {len(order) + 1} pages)')

In [ ]:
# Cell 8 - ONNX export + 4-decimal parity check
import torch.nn as nn
import onnxruntime as ort

class CwsbOnnx(nn.Module):
    """Input: 384x384x3 RGB uint8 (already resized + centre-cropped outside the graph).
    Output: probabilities (1,2) = [healthy, wsb] and feature_map (1,768,12,12) for heatmaps."""
    def __init__(self, core):
        super().__init__()
        self.core = core
        self.register_buffer('mean', torch.tensor(MEAN).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(STD).view(1, 3, 1, 1))
    def forward(self, rgb_uint8):
        x = rgb_uint8.permute(2, 0, 1).unsqueeze(0).float() / 255.0
        x = (x - self.mean) / self.std
        feat = self.core.forward_features(x)
        logits = self.core.forward_head(feat)
        return F.softmax(logits, dim=1), feat

cpu_model = new_model.to('cpu').eval()
onnx_path = f'models/micro/{TAG}.onnx'
torch.onnx.export(CwsbOnnx(cpu_model).eval(),
                  torch.randint(0, 256, (IMG, IMG, 3), dtype=torch.uint8), onnx_path,
                  input_names=['rgb_384'], output_names=['probabilities', 'feature_map'],
                  opset_version=17, dynamo=False)

norm_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
worst = 0.0
for r in te[:10]:                                   # real test images, not random noise
    crop = np.asarray(disp_tf(Image.open(Path('work/clean') / r['file']).convert('RGB')), dtype=np.uint8)
    with torch.no_grad():
        ref = torch.softmax(cpu_model(norm_tf(Image.fromarray(crop)).unsqueeze(0)), 1).numpy()
    got = sess.run(['probabilities'], {'rgb_384': crop})[0]
    worst = max(worst, float(np.max(np.abs(ref - got))))
print(f'max |pytorch - onnx| over {min(len(te), 10)} test images: {worst:.2e}')
assert worst < 1e-4, f'parity FAILED ({worst})'
print(f'Parity PASSED -> {onnx_path}  ({os.path.getsize(onnx_path) / 1e6:.1f} MB)')

In [ ]:
# Cell 9 - package everything as the versioned release and download
rel = Path(f'/content/{TAG}_release')
shutil.rmtree(rel, ignore_errors=True)
rel.mkdir()
shutil.copy('models/micro/model.pt', rel / f'{TAG}.pt')
shutil.copy(onnx_path, rel / f'{TAG}.onnx')
shutil.copy('models/micro/config.json', rel / f'config_micro_v{VERSION_NUMBER}.json')
shutil.copy('models/micro/training_result.json', rel / f'training_result_micro_v{VERSION_NUMBER}.json')
shutil.copy('work/splits/split.csv', rel / f'split_micro_v{VERSION_NUMBER}.csv')
shutil.copy('work/clean/images.csv', rel / f'quality_micro_v{VERSION_NUMBER}.csv')
if pdf_path.exists():
    shutil.copy(pdf_path, rel / pdf_path.name)      # the heatmap evidence PDF

zip_name = f'{TAG}_release.zip'
with zipfile.ZipFile('/content/' + zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(rel.iterdir()):
        zf.write(p, arcname=p.name)
print(f'{zip_name}: ' + ', '.join(p.name for p in sorted(rel.iterdir())))
from google.colab import files as gfiles
gfiles.download('/content/' + zip_name)

## After downloading

1. Unzip **flat** into `models/micro_v1/`, with `cwsb_micro_v1.pt` directly inside.
2. Read Cell 3's quality table. If many good images were flagged, or bad ones were not, adjust
   the limits in Cell 1; switch `QUALITY_GATE` to `'exclude'` only once the limits fit.
3. Read Cell 6. The **plants** line is how the device decides (highest score among a plant's
   images). Then look at the first pages of the PDF: they are the wrong and inconclusive images.
4. Check the heatmaps. The red area should sit on the hole, ridge or frass. If it sits on the
   image border, the lighting ring or the background, the model has learned a shortcut and more
   varied data is needed.
5. **Keep for the next retrain:** the whole upload folder, plus `cwsb_micro_v1.pt`,
   `cwsb_micro_v1.onnx` and `split_micro_v1.csv` from the release zip.